# Finance Mart — execute current Databricks parity checks

Prepared locally; **Databricks execution is NOT_RUN until this notebook completes**. Use only the existing Free Edition serverless compute; no paid upgrade is needed or authorized.

Run all cells. This creates three uniquely named `workspace.finance_verify_*` schemas, builds the current SQL port over two raw edge-case fixtures and a snapshot of `workspace.finance_mart.raw_lines`, and retains the result tables. It does not update or delete the original source or historical mart. Full-data row comparisons cover dimensions and marts; large staging/fact relations are checked by counts, exact financial totals and verdict groups. Source identity is checked by eight commutative SHA-256 chunk sums plus count/IDs, not a byte-for-byte cloud CSV comparison. Decimal money is exact; floating ratios allow absolute tolerance 1e-9 and relative tolerance 1e-12.

The DuckDB reference first passed independent hand-calculated return/cohort oracles. Download the final JSON and return it for publication. A local reference or parsed notebook does not prove hosted parity.

In [ ]:
import base64
import hashlib
import json
import math
import uuid
import zlib
from datetime import date, datetime, timezone
from decimal import Decimal

encoded_reference = (
    '...'
)
payload = json.loads(zlib.decompress(base64.b64decode(encoded_reference)))
RAW_COLUMNS = ['line_id', 'Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer_ID', 'Country']
SUMMARY_SQL = (
    "select 'gross_sales' as metric, coalesce(sum(amount), 0) as value from fact_lines where line_type='sale'\n"
    "union all select 'returns', coalesce(sum(amount), 0) from fact_lines where line_type='return'\n"
    "union all select 'quick_reversals', coalesce(sum(amount), 0) from fact_lines where is_quick_reversal\n"
    "union all select 'kept_amount', coalesce(sum(amount), 0) from fact_lines\n"
    "union all select 'quarantined_amount', coalesce(sum(amount), 0) from quarantine_lines\n"
)
def split_sql(text):
    """Split this port's SQL without breaking quoted semicolons or comments."""
    statements, chunk, quote, comment, i = [], [], None, None, 0
    while i < len(text):
        char, following = text[i], text[i:i + 2]
        if comment == "line":
            if char == "\n":
                comment = None
                chunk.append(char)
        elif comment == "block":
            if following == "*/":
                comment = None
                i += 1
                chunk.append(" ")
        elif quote:
            chunk.append(char)
            if char == "\\" and i + 1 < len(text):
                i += 1
                chunk.append(text[i])
            elif char == quote:
                if i + 1 < len(text) and text[i + 1] == quote:
                    i += 1
                    chunk.append(text[i])
                else:
                    quote = None
        elif following in ("--", "/*"):
            comment = "line" if following == "--" else "block"
            i += 1
            chunk.append(" ")
        elif char in "'\"`":
            quote = char
            chunk.append(char)
        elif char == ";":
            if "".join(chunk).strip():
                statements.append("".join(chunk).strip())
            chunk = []
        else:
            chunk.append(char)
        i += 1
    if quote or comment == "block":
        raise ValueError("Unterminated SQL quote/comment")
    if "".join(chunk).strip():
        statements.append("".join(chunk).strip())
    return statements


def scalar(value):
    if value is None or isinstance(value, (str, bool, int)):
        return value
    if isinstance(value, (date, datetime)):
        return value.isoformat(sep=" ") if isinstance(value, datetime) else value.isoformat()
    if isinstance(value, (float, Decimal)):
        if not math.isfinite(float(value)):
            raise ValueError("Non-finite result")
        return str(value)
    raise TypeError(type(value).__name__)


def signature_sql(engine):
    columns = ['cast(line_id as string)' if engine == "spark" else 'cast(line_id as varchar)']
    columns += [f'`{name}`' if engine == "spark" else f'"{name.replace("_", " ")}"' for name in RAW_COLUMNS[1:]]
    if engine == "spark":
        digest = "sha2(to_json(array(" + ",".join(columns) + ")),256)"
        chunks = [f"sum(cast(conv(substr(h,{1 + i * 8},8),16,10) as decimal(38,0))) as h{i}" for i in range(8)]
    else:
        digest = "sha256(cast(json_array(" + ",".join(columns) + ") as varchar))"
        chunks = [f"sum(cast('0x' || substr(h,{1 + i * 8},8) as bigint)) as h{i}" for i in range(8)]
    return ("select count(*) as rows, count(distinct line_id) as distinct_ids, min(line_id) as min_id, "
            "max(line_id) as max_id, " + ",".join(chunks) + " from (select line_id," + digest + " as h from raw_lines)")


def compare_rows(expected, actual):
    """Multiset comparison: exact decimals/integers/nulls, bounded floating tolerance."""
    types = expected["types"]

    def normalized(row):
        values = []
        for kind, value in zip(types, row, strict=True):
            if value is None:
                values.append((0, ""))
            elif kind in ("FLOAT", "DOUBLE", "REAL"):
                values.append((1, float(value)))
            elif kind.startswith("DECIMAL") or "INT" in kind or kind == "HUGEINT":
                values.append((1, Decimal(str(value))))
            else:
                values.append((1, str(value)))
        return values

    left, right = sorted(map(normalized, expected["rows"])), sorted(map(normalized, actual))
    if len(left) != len(right):
        return False
    for old, new in zip(left, right, strict=True):
        for kind, a, b in zip(types, old, new, strict=True):
            if a[0] != b[0]:
                return False
            if a[0] and kind in ("FLOAT", "DOUBLE", "REAL"):
                if not math.isclose(a[1], b[1], rel_tol=1e-12, abs_tol=1e-9):
                    return False
            elif a != b:
                return False
    return True


def runtime_body():
    # Executed only by the user in an existing Databricks notebook session.
    spark.sql("SET TIME ZONE 'UTC'")  # noqa: F821
    run_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:8]  # noqa: F821
    evidence = {
        "run_id": run_id, "executed_at_utc": datetime.now(timezone.utc).isoformat(),
        "engine": "Databricks Spark", "spark_version": spark.version,  # noqa: F821
        "reference_base_commit": payload["base_commit"], "code_lf_sha256": payload["code_lf_sha256"],  # noqa: F821
        "reference_source_sha256": payload["source_sha256"], "checks": [], "schemas": [], "scenarios": {},  # noqa: F821
        "scope": "Synthetic raw fixtures plus historical public retail data. No production or performance claim.",
    }

    def record(name, passed, observed=None):
        evidence["checks"].append({"name": name, "passed": bool(passed), "observed": observed})
        if not passed:
            raise AssertionError(name)

    try:
        for name, reference in payload["scenarios"].items():  # noqa: F821
            schema = "workspace.finance_verify_" + run_id + "_" + name
            spark.sql("CREATE SCHEMA " + schema)  # noqa: F821
            evidence["schemas"].append(schema)
            spark.sql("USE SCHEMA " + schema)  # noqa: F821
            if name == "full":
                # One CTAS freezes the existing uploaded source; nothing overwrites it.
                spark.sql("CREATE TABLE raw_lines AS SELECT * FROM workspace.finance_mart.raw_lines")  # noqa: F821
            else:
                definitions = "line_id LONG," + ",".join(f"`{col}` STRING" for col in RAW_COLUMNS[1:])
                rows = [(i + 1, *row) for i, row in enumerate(payload["fixture_rows"][name])]  # noqa: F821
                spark.createDataFrame(rows, definitions).write.mode("errorifexists").saveAsTable(schema + ".raw_lines")  # noqa: F821
            actual_signature = [scalar(x) for x in spark.sql(signature_sql("spark")).first()]  # noqa: F821
            # Numeric JSON representation may use decimal scale; compare numerical values.
            record(name + ":frozen_raw_signature", all(
                Decimal(str(a)) == Decimal(str(b)) for a, b in zip(actual_signature, reference["raw_signature"], strict=True)
            ), actual_signature)
            for file, sql in payload["port_sql"].items():  # noqa: F821
                record(name + ":source:" + file,
                       hashlib.sha256(sql.encode()).hexdigest() == payload["code_lf_sha256"][file])  # noqa: F821
                statements = split_sql(sql)
                for statement in statements:
                    spark.sql(statement).collect()  # noqa: F821
                evidence["scenarios"].setdefault(name, {})[file] = len(statements)
            for table, count in reference["counts"].items():
                observed = spark.sql("SELECT count(*) FROM " + table).first()[0]  # noqa: F821
                record(name + ":count:" + table, observed == count, observed)
            amounts = {r[0]: scalar(r[1]) for r in spark.sql(SUMMARY_SQL).collect()}  # noqa: F821
            record(name + ":exact_money", all(Decimal(amounts[k]) == Decimal(v) for k, v in reference["amounts"].items()), amounts)
            verdicts = {r[0]: r[1] for r in spark.sql("SELECT verdict,count(*) FROM line_classified GROUP BY verdict").collect()}  # noqa: F821
            record(name + ":classification", verdicts == reference["verdicts"], verdicts)
            for table, expected in reference["rows"].items():
                frame = spark.table(table).select(*expected["columns"])  # noqa: F821
                actual = [[scalar(value) for value in row] for row in frame.collect()]
                record(name + ":rows:" + table, compare_rows(expected, actual), {"rows": len(actual), "columns": expected["columns"]})
        evidence["status"] = "PASS"
    except Exception as exc:
        evidence["status"] = "FAIL"
        evidence["error"] = str(exc)[:2000]
        raise
    finally:
        result = json.dumps(evidence, indent=2)
        print(result)
        encoded = base64.b64encode(result.encode()).decode()
        displayHTML(  # noqa: F821
            '<h2>Finance validation: ' + evidence["status"] + '</h2><p>Download the actual executed evidence.</p>'
            '<a download="finance-databricks-' + run_id + '.json" href="data:application/json;base64,' + encoded
            + '">Download validation JSON</a><p>The isolated schemas are retained for inspection. '
            'No source table or historical report was overwritten.</p>'
        )


In [ ]:
runtime_body()
